#### Demo - Output Types [Data Class and Pydantic]

In [ ]:
# import libraries
# Imports environment variables from a `.env` file.
from dotenv import load_dotenv

# Imports Agent, Runner, trace (for logging), and function_tool (for custom tools) from the agents module.
from agents import Agent, Runner, trace


In [3]:

load_dotenv()

True

##### Travel Agent using Data Class Output_type

In [ ]:
# For defining data class
from dataclasses import dataclass
# Define the structured output schema (dataclass)
@dataclass
class TravelRecommendation:
    destination: str
    reason: str
    top_tip: str

In [ ]:
# Create the Agent and enforce structured output with output_type
travel_agent = Agent(
    name="Travel Recommender",
    instructions=(
        "You are a travel recommendation agent. "
        "ALWAYS return exactly one travel recommendation with:\n"
        "- destination: a specific city/region + country\n"
        "- reason: short, practical justification\n"
        "- top_tip: one actionable tip (timing, local etiquette, transport, or booking)\n"
        "Be concise and avoid extra fields."
    ),
    model="gpt-5-mini",
    output_type=TravelRecommendation,  # Enforces destination/reason/top_tip
)

In [7]:
# Run the Agent
user_message = "I want a relaxing 3-day trip in January with good food and walkable neighborhoods."
result = await Runner.run(travel_agent, user_message)
print(result.final_output)

TravelRecommendation(destination='Lisbon, Portugal', reason='Mild January weather, compact and walkable neighborhoods (Baixa, Chiado, Alfama) with excellent seafood, bakeries and relaxed cafés—perfect for a 3-day chill food-focused trip.', top_tip='Book dinner reservations 1–2 days ahead (aim for 19:00–20:00), as the best local restaurants are small and fill up quickly.')


In [ ]:
# Accessing Specific Properties of the Structured Output
rec: TravelRecommendation = result.final_output  # typed dataclass instance
print("Destination:", rec.destination)
print("Reason:", rec.reason)
print("Top tip:", rec.top_tip)

Destination: Lisbon, Portugal
Reason: Mild January weather, compact and walkable neighborhoods (Baixa, Chiado, Alfama) with excellent seafood, bakeries and relaxed cafés—perfect for a 3-day chill food-focused trip.
Top tip: Book dinner reservations 1–2 days ahead (aim for 19:00–20:00), as the best local restaurants are small and fill up quickly.


##### Travel Agent using Pydantic obejct Output_type

In [9]:
from pydantic import BaseModel
# 1) Define the structured output schema (Pydantic)
class TravelRecommendation(BaseModel):
    destination: str 
    reason: str 
    top_tip: str 


In [ ]:
# 2) Create the Agent and enforce structured output with output_type
travel_agent = Agent(
    name="Travel Recommender",
    instructions=(
        "You are a travel recommendation agent. "
        "ALWAYS return exactly one travel recommendation with:\n"
        "- destination: a specific city/region + country\n"
        "- reason: short, practical justification\n"
        "- top_tip: one actionable tip (timing, local etiquette, transport, or booking)\n"
        "Be concise and avoid extra fields."
    ),
    model="gpt-5-mini",
    output_type=TravelRecommendation,  # Pydantic model
)

In [11]:
result = await Runner.run(
    travel_agent,
    "I want a relaxing 3-day trip in January with good food and walkable neighborhoods."
)

In [12]:
rec: TravelRecommendation = result.final_output  # Pydantic object
print("Destination:", rec.destination)
print("Reason:", rec.reason)
print("Top tip:", rec.top_tip)

Destination: Lisbon, Portugal
Reason: Mild January weather, compact walkable neighborhoods (Chiado/Alfama/Bairro Alto) and outstanding seafood, cafés and pastries—ideal for a relaxed 3‑day food-and-walk trip.
Top tip: Book dinners (seafood tascas and fado houses) 1–2 days in advance to avoid long waits.


In [13]:
# Optional: view as dict / JSON
print(rec.model_dump())         # dict
print(rec.model_dump_json())    # JSON string

{'destination': 'Lisbon, Portugal', 'reason': 'Mild January weather, compact walkable neighborhoods (Chiado/Alfama/Bairro Alto) and outstanding seafood, cafés and pastries—ideal for a relaxed 3‑day food-and-walk trip.', 'top_tip': 'Book dinners (seafood tascas and fado houses) 1–2 days in advance to avoid long waits.'}
{"destination":"Lisbon, Portugal","reason":"Mild January weather, compact walkable neighborhoods (Chiado/Alfama/Bairro Alto) and outstanding seafood, cafés and pastries—ideal for a relaxed 3‑day food-and-walk trip.","top_tip":"Book dinners (seafood tascas and fado houses) 1–2 days in advance to avoid long waits."}


#### Various ways to define Pydantic Object

Field(...) in Pydantic is used to add extra meaning and rules to a model attribute. In the syntax Field(..., description="Specific city/region + country"), the ellipsis (...) indicates that the field is required and has no default value, while the description provides human- and model-readable context about what the field should contain. This metadata helps the SDK and the language model generate correct structured output and allows Pydantic to validate that the required field is present and correctly formed.

In [ ]:

from pydantic import BaseModel, Field
class TravelRecommendation(BaseModel):
    destination: str = Field(..., description="Specific city/region + country")
    reason: str = Field(..., description="Short practical justification")
    top_tip: str = Field(..., description="One actionable travel tip")

In [15]:
from pydantic import BaseModel, Field

class SupportTicket(BaseModel):
    issue_title: str
    issue_description: str
    customer_id: str

    priority: str = Field("Medium", description="Low, Medium, High")
    assigned_team: str | None = None
    status: str = "Open"


| Field             | Required?            |
| ----------------- | -------------------- |
| issue_title       | ✅                    |
| issue_description | ✅                    |
| customer_id       | ✅                    |
| priority          | ❌ (default = Medium) |
| assigned_team     | ❌                    |
| status            | ❌ (default = Open)   |
